# Сбор данных котировок и новостей

Источники: **MOEX ISS** (котировки, часовые свечи), **Интерфакс** (новости), **Банк России** (публикации и решения по ставке).

## 1. Настройка

Импорты, период собираемых данных и папки для файлов.

In [1]:
from pathlib import Path
from urllib.parse import urljoin
import re
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup

DATE_START = "2025-10-24"
DATE_END = "2026-10-23"  # последний день с торгами в выбранном периоде
TZ = "Europe/Moscow"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

STOCKS_DIR = DATA_DIR / "stocks_by_ticker"
STOCKS_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0"}

## 2. Котировки MOEX

19 акций пяти секторов. Выбраны только интересующие меня скучные базированые сектора: нефтегаз, металлы, добыча, удобрения и энергетика.

Получаем часовые свечи из режима торгов `TQBR` через ISS API. 

In [2]:
SECTORS = {
    "oil_gas": ["GAZP", "LKOH", "ROSN", "NVTK", "TATN", "SNGS"],
    "metals": ["GMKN", "CHMF", "NLMK", "MAGN", "RUAL"],
    "mining": ["PLZL", "SELG", "ALRS"],
    "fertilizers": ["PHOR", "AKRN"],
    "energy": ["IRAO", "HYDR", "UPRO"],
}

CANDLE_INTERVAL = 60  # минут
BOARD = "TQBR"

### 2.1. Функция загрузки

MOEX отдаёт данные кусками: параметр `start` сдвигает начало следующей страницы.

In [3]:
def get_candles(ticker):
    url = (
        "https://iss.moex.com/iss/engines/stock/markets/shares/"
        f"boards/{BOARD}/securities/{ticker}/candles.json"
    )
    params = {
        "from": DATE_START,
        "till": DATE_END,
        "interval": CANDLE_INTERVAL,
        "start": 0,
        "iss.meta": "off",
    }
    all_rows = []
    columns = None

    while True:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        block = response.json()["candles"]
        columns, rows = block["columns"], block["data"]
        if not rows:
            break
        all_rows.extend(rows)
        params["start"] += len(rows)
        time.sleep(0.3)

    df = pd.DataFrame(all_rows, columns=columns)
    if not df.empty:
        df = df.drop_duplicates("begin").sort_values("begin").reset_index(drop=True)
    df["ticker"] = ticker
    return df

### 2.2. Выгрузка и объединение

Каждый тикер сохраняем отдельно. При повторном запуске используем локальный файл.

In [4]:
stock_frames = []
stock_errors = []

for sector, tickers in SECTORS.items():
    for ticker in tickers:
        path = STOCKS_DIR / f"{ticker}_1h.csv"
        try:
            if path.exists():
                df = pd.read_csv(path)
            else:
                df = get_candles(ticker)
                if df.empty:
                    raise ValueError("Нет свечей")
                df["sector"] = sector
                df.to_csv(path, index=False, encoding="utf-8-sig")

            df["ticker"] = ticker
            df["sector"] = sector
            stock_frames.append(df)
            print(f"{ticker}: {len(df)} свечей")
        except (requests.RequestException, ValueError, KeyError) as exc:
            stock_errors.append((ticker, str(exc)))
            print(f"Ошибка {ticker}: {exc}")

if not stock_frames:
    raise RuntimeError("Не удалось получить ни одной акции")

stocks = (
    pd.concat(stock_frames, ignore_index=True)
    .drop_duplicates(["ticker", "begin"])
    .sort_values(["ticker", "begin"])
    .reset_index(drop=True)
)
stocks.to_csv(DATA_DIR / "stocks.csv", index=False, encoding="utf-8-sig")

print(f"Итого: {stocks['ticker'].nunique()} акций, {len(stocks)} свечей")
print("Ошибки:", stock_errors)

GAZP: 5362 свечей
LKOH: 5359 свечей
ROSN: 5359 свечей
NVTK: 5362 свечей
TATN: 5358 свечей
SNGS: 5334 свечей
GMKN: 5362 свечей
CHMF: 5360 свечей
NLMK: 5355 свечей
MAGN: 5361 свечей
RUAL: 5359 свечей
PLZL: 5339 свечей
SELG: 5329 свечей
ALRS: 5361 свечей
PHOR: 5337 свечей
AKRN: 4755 свечей
IRAO: 5362 свечей
HYDR: 5305 свечей
UPRO: 5289 свечей
Итого: 19 акций, 101008 свечей
Ошибки: []


### 2.3. Техническая проверка

Сверяем диапазоны дат, дубликаты и пустые значения. Допускаем разное количество свеч

In [5]:
stock_check = stocks.groupby("ticker").agg(
    candles=("begin", "size"),
    first=("begin", "min"),
    last=("begin", "max"),
)
display(stock_check.sort_values("candles"))
print("Дубликаты:", stocks.duplicated(["ticker", "begin"]).sum())
print("Пустые значения:", stocks.isna().sum().sum())

,candles,first,last
ticker,,,
AKRN,4755,2025-10-24 07:00:00,2026-10-23 23:00:00
UPRO,5289,2025-10-24 06:00:00,2026-10-23 23:00:00
HYDR,5305,2025-10-24 06:00:00,2026-10-23 23:00:00
SELG,5329,2025-10-24 07:00:00,2026-10-23 23:00:00
SNGS,5334,2025-10-24 06:00:00,2026-10-23 23:00:00
PHOR,5337,2025-10-24 06:00:00,2026-10-23 23:00:00
PLZL,5339,2025-10-24 06:00:00,2026-10-23 23:00:00
NLMK,5355,2025-10-24 06:00:00,2026-10-23 23:00:00
TATN,5358,2025-10-24 06:00:00,2026-10-23 23:00:00


Дубликаты: 0
Пустые значения: 0


## 3. Интерфакс

У Интерфакса есть архив новостей, можно найти по дате. Ищем.

Парсим архив по дням: `datetime`, `title`, `url`, `source`. Полные тексты не собираем.

### 3.1. Парсер одного дня

пример HTML-элемента из новостного архива:

```html
<div class="news-item">
    <span class="time">11:34</span>
    <a href="/business/1054414">
        Текст заголовка новости
    </a>
</div>
```

Основные элементы следующие:

- `<div>` — контейнер, объединяющий элементы новости.
- `<span>` — элемент, содержащий время публикации.
- `<a>` — ссылка на новость.
- `href` — атрибут с адресом статьи.
- `11:34` — время публикации.
- `Текст заголовка новости` — заголовок.


Собственно методом ниже находим ссылки на публикации и время в HTML-блоке вокруг заголовка.

In [6]:
def get_interfax_news(date):
    date = pd.Timestamp(date)
    archive_url = f"https://www.interfax.ru/business/news/{date:%Y/%m/%d}/"
    response = requests.get(archive_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    records = []
    for a in soup.find_all("a", href=True):
        link = urljoin(archive_url, a["href"])

        # проверяем соответствие ссылки формату статьи интерфакса
        if not re.fullmatch(r"https://www\.interfax\.ru/[a-z-]+/\d+/?", link):
            continue

        # проверяем наличие родительского элемента
        if a.parent is None:
            continue

        # ищем время публикации в родительском элементе. Если нет, пропускаем
        match = re.match(r"^(\d{1,2}:\d{2})\s+", a.parent.get_text(" ", strip=True))
        if not match:
            continue
        
        title = a.get_text(" ", strip=True)
        if title:
            records.append({
                "datetime": f"{date:%Y-%m-%d} {match.group(1)}",
                "title": title,
                "url": link,
                "source": "Interfax",
            })

    return pd.DataFrame(
        records, columns=["datetime", "title", "url", "source"]
    ).drop_duplicates("url")

### 3.2. Годовой архив

Сохраняем каждый день отдельно; неудачные даты можно загрузить повторным запуском.

In [7]:
INTERFAX_DIR = DATA_DIR / "interfax_daily"
INTERFAX_DIR.mkdir(exist_ok=True)

interfax_errors = []
for date in pd.date_range(DATE_START, DATE_END, freq="D"):
    day = date.strftime("%Y-%m-%d")
    path = INTERFAX_DIR / f"{day}.csv"
    if path.exists():
        continue
    try:
        df_day = get_interfax_news(day)
        df_day.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"{day}: {len(df_day)}")
    except requests.RequestException as exc:
        interfax_errors.append((day, str(exc)))
        print(f"Ошибка {day}: {exc}")
    time.sleep(1.5)

print("Ошибок:", len(interfax_errors))

Ошибок: 0


### 3.3. Общий CSV

Объединяем дневные файлы в один большой csv.

In [8]:
interfax_files = sorted(INTERFAX_DIR.glob("*.csv"))

# проверка что есть все дневные файлы, все дни загружены
assert len(interfax_files) == len(pd.date_range(DATE_START, DATE_END)), (
    "Не все дни загружены"
)

news = pd.concat(
    [pd.read_csv(path) for path in interfax_files], ignore_index=True
)
news = news.drop_duplicates("url")
news["datetime"] = pd.to_datetime(news["datetime"])
news = news.sort_values("datetime").reset_index(drop=True)
news.to_csv(DATA_DIR / "interfax_news.csv", index=False, encoding="utf-8-sig")

print(f"Дней: {len(interfax_files)}; новостей: {len(news)}")
print("Диапазон:", news["datetime"].min(), "—", news["datetime"].max())

Дней: 365; новостей: 14220
Диапазон: 2025-10-24 00:03:00 — 2026-10-23 22:31:00


### 3.4. Попытка в фильтрацию по темам

Очень грубо (с применением тематических слов и регулярок) пытаемся отфильтровать только нужные новости

In [9]:
PATTERNS = {
    "oil_gas": [
        r"\b(?:газпром|лукойл|роснефт|новатэк|татнефт|сургутнефт)\w*",
        r"\bнефт\w*",
        r"\bгаз(?:а|у|ом|е|ов\w*|опровод\w*)\b",
        r"\b(?:опек|brent|нпз)\b",
        r"\bуглеводород\w*",
        r"\bсахалинск\w*\s+энерги\w*",
    ],
    "metals": [
        r"\b(?:норникел|нлмк|северстал|ммк|русал)\w*",
        r"\b(?:металлург|алюмин|никел)\w*",
        r"\b(?:сталь|стали|стальн\w*)\b",
        r"\b(?:медь|меди|медн\w*)\b",
    ],
    "mining": [
        r"\b(?:полюс|селигдар|алрос)\w*",
        r"\b(?:золот|алмаз|паллади|платин)\w*",
    ],
    "fertilizers": [
        r"\b(?:фосагро|акрон|удобрени|фосфат|калийн|калий)\w*",
    ],
    "energy": [
        r"\b(?:интер\s+рао|русгидро|юнипро)\b",
        r"\b(?:электроэнерг|электростанц|электросет)\w*",
        r"\b(?:энергосистем|энергоснабж|энерготариф)\w*",
    ],
    "macro": [
        r"\bключев\w*\s+ставк\w*",
        r"\b(?:цб|банк\w*\s+россии)\b.*\b(?:ставк|дкп|дку|инфляц|проинфляц|денежно-кредитн)\w*",
        r"\bставк\w*.*\bцб\b",
        r"\bинфляц\w*.*\b(?:рф|росси\w*)\b",
        r"\b(?:рф|росси\w*)\b.*\bинфляц\w*",
        r"\bсанкц\w*",
        r"\bмеждународн\w*\s+резерв\w*",
        r"\bденежн\w*\s+масс\w*",
        r"\bрубл\w*.*\b(?:юан\w*|доллар\w*|евро|курс\w*|укреп\w*|ослаб\w*|сниж\w*|подешев\w*)",
        r"\b(?:курс\w*|юан\w*|доллар\w*|евро).*?\bрубл\w*",
    ],
}

COMPILED_PATTERNS = {
    topic: [re.compile(pattern, re.I) for pattern in rules]
    for topic, rules in PATTERNS.items()
}

In [10]:
def classify_news(title):
    title = str(title)
    matches = []

    for topic, rules in COMPILED_PATTERNS.items():
        if any(rule.search(title) for rule in rules):
            matches.append(topic)

    return ", ".join(matches)

interfax_filtered = news.copy()
interfax_filtered["topics"] = interfax_filtered["title"].apply(classify_news)
interfax_filtered = interfax_filtered[interfax_filtered["topics"] != ""].copy()
interfax_filtered.to_csv(
    DATA_DIR / "interfax_filtered.csv", index=False, encoding="utf-8-sig"
)
print(f"Отобрано: {len(interfax_filtered)} из {len(news)}")
display(interfax_filtered[["title", "topics"]].head(5))

Отобрано: 3727 из 14220


,title,topics
0,"ЦБ рассмотрит вопрос о ставке, большинство ана...",macro
3,"Нефть Brent подешевела до $65,73 за баррель",oil_gas
5,"ФАС выдала предупреждение ""дочке"" ""Газпром неф...",oil_gas
10,Рубль почти не меняется в паре с юанем перед р...,macro
13,"В ВТБ ожидают ""очень сложного"" заседания ЦБ РФ...",macro


В результат ячейки сверху это не попадет, но во время игр с новостями было обнаружено, что могут в фильтры попадаться невалидные новости (например про золотой купол залетала в тематику золота). Оставляем на потом.

## 4. Банк России

С Банка РФ будем собирать решения по ставке из архива и другие публикации.
Решения по ставке будем парсить из HTML-архива решений по ставке. 
Другие публикации через официальный SOAP-сервис `NewsInfoXML`


### 4.1. Решения по ключевой ставке

В архивных ссылках время решения зашито в имени файла. Сохраняем отдельный CSV.

In [11]:
RATE_FILE = DATA_DIR / "cbr_key_rate.csv"

if RATE_FILE.exists():
    df_rates = pd.read_csv(RATE_FILE)
else:
    archive_url = "https://www.cbr.ru/dkp/mp_dec/decision_key_rate/"
    response = requests.get(archive_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    records = []

    for a in soup.find_all("a", href=True):
        link = urljoin(archive_url, a["href"])
        match = re.search(r"file=(\d{8}_\d{6})key\.htm", link, re.I)
        if not match:
            continue
        dt = pd.to_datetime(match.group(1), format="%d%m%Y_%H%M%S").tz_localize(TZ)
        if not (
            pd.Timestamp(DATE_START, tz=TZ)
            <= dt
            < pd.Timestamp(DATE_END, tz=TZ) + pd.Timedelta(days=1)
        ):
            continue
        records.append({
            "datetime": dt, "title": a.get_text(" ", strip=True),
            "url": link, "source": "CBR", "topics": "macro",
            "event_type": "key_rate_decision",
        })

    df_rates = pd.DataFrame(records)
    df_rates = df_rates.drop_duplicates("url").sort_values("datetime")
    df_rates.to_csv(RATE_FILE, index=False, encoding="utf-8-sig")

print("Решений по ставке:", len(df_rates))

Решений по ставке: 9


### 4.2. Другие публикации

Как говорилось выше, будем использовать специальный сервис ЦБ.

Пример XML-ответа сервиса `NewsInfoXML`:

```xml
<NewsInfoXMLResult>
    <News>
        <DocDate>2026-10-23T13:30:00+03:00</DocDate>
        <Title>Пример сообщения Банка России</Title>
        <Url>/press/event/?id=12345</Url>
        <Doc_id>12345</Doc_id>
    </News>
</NewsInfoXMLResult>
```

Основные элементы:

- `<NewsInfoXMLResult>` — блок с результатом запроса.
- `<News>` — отдельная публикация.
- `<DocDate>` — дата и время документа.
- `<Title>` — заголовок публикации.
- `<Url>` — ссылка на публикацию.
- `<Doc_id>` — идентификатор записи.

Для парсинга используем библиотеку ElementTree

In [12]:
def get_cbr_news(date_from, date_to):

    # собираем запрос
    soap_body = f'''<?xml version="1.0" encoding="utf-8"?>
    <soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
      <soap:Body>
        <NewsInfoXML xmlns="http://web.cbr.ru/">
          <fromDate>{date_from}</fromDate>
          <ToDate>{date_to}</ToDate>
        </NewsInfoXML>
      </soap:Body>
    </soap:Envelope>'''

    # отправляем, получаем ответ
    response = requests.post(
        "https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx",
        data=soap_body.encode("utf-8"),
        headers={
            "Content-Type": "text/xml; charset=utf-8",
            "SOAPAction": '"http://web.cbr.ru/NewsInfoXML"',
        },
        timeout=60,
    )
    response.raise_for_status()
    root = ET.fromstring(response.content)

    # проверяем ответ на ошибки
    if root.find(".//{*}Fault") is not None:
        raise ValueError("SOAP вернул ошибку")
    result = root.find(".//{*}NewsInfoXMLResult")

    # проверяем есть ли новость
    if result is None:
        raise ValueError("В ответе нет NewsInfoXMLResult")

    records = []
    for item in result.findall(".//{*}News"):
        records.append({
            "datetime": item.findtext("{*}DocDate"),
            "title": item.findtext("{*}Title"),
            "url": urljoin("https://www.cbr.ru/", item.findtext("{*}Url") or ""),
            "source": "CBR",
            "doc_id": item.findtext("{*}Doc_id"),
        })
    return pd.DataFrame(records, columns=["datetime", "title", "url", "source", "doc_id"])

### 4.3. Выгрузка по месяцам

Собираем новости по месяцам

In [13]:
CBR_DIR = DATA_DIR / "cbr_monthly"
CBR_DIR.mkdir(exist_ok=True)

cbr_errors = []
for month in pd.period_range(DATE_START, DATE_END, freq="M"):
    start = max(month.start_time.normalize(), pd.Timestamp(DATE_START))
    end = min(month.end_time.normalize(), pd.Timestamp(DATE_END))
    path = CBR_DIR / f"{month}.csv"
    if path.exists():
        continue
    try:
        df_month = get_cbr_news(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
        df_month.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"{month}: {len(df_month)} публикаций")
    except (requests.RequestException, ValueError, ET.ParseError) as exc:
        cbr_errors.append((str(month), str(exc)))
        print(f"Ошибка {month}: {exc}")
    time.sleep(1.5)

print("Ошибки:", cbr_errors)

Ошибки: []


### 4.4. Исправление дат и единый CSV ЦБ

Замечено, что дата `DocDate` встречается в разных форматах. Приводим всё к московскому времени. Кроме того удаляем повторы. Эмпирическим путем пришли к тому что удалять надо **по `doc_id`**, а не по URL.

In [14]:
def parse_cbr_date(value):
    if pd.isna(value):
        return pd.NaT
    try:
        dt = pd.to_datetime(value, format="mixed")
        return dt.tz_localize(TZ) if dt.tzinfo is None else dt.tz_convert(TZ)
    except (ValueError, TypeError):
        return pd.NaT

cbr_files = sorted(CBR_DIR.glob("*.csv"))
assert len(cbr_files) == len(pd.period_range(DATE_START, DATE_END, freq="M")), (
    "Не все месяцы загружены. Повторите предыдущую ячейку."
)

cbr_news = pd.concat(
    [pd.read_csv(path, dtype={"datetime": "string", "doc_id": "string"}) for path in cbr_files],
    ignore_index=True,
)
cbr_news["datetime"] = cbr_news["datetime"].apply(parse_cbr_date)
cbr_news = cbr_news.drop_duplicates("doc_id").sort_values("datetime").reset_index(drop=True)

assert cbr_news["datetime"].notna().all(), "Есть непарсенные даты ЦБ"
assert cbr_news["doc_id"].notna().all(), "Есть публикации без doc_id"
cbr_news.to_csv(DATA_DIR / "cbr_news.csv", index=False, encoding="utf-8-sig")

print("Публикаций ЦБ:", len(cbr_news))
print("Диапазон:", cbr_news["datetime"].min(), "—", cbr_news["datetime"].max())

Публикаций ЦБ: 1121
Диапазон: 2025-10-27 08:30:02.813000+03:00 — 2026-10-22 19:48:16.627000+03:00


## 5. Итоговая таблица новостей

Объединяем два источника. Для ЦБ используем `doc_id`: разные публикации могут вести на один URL.

In [15]:
cbr = cbr_news.copy()
rates = df_rates.copy()
interfax = interfax_filtered.copy()

# Все временные метки приводим к Europe/Moscow.
cbr["datetime"] = pd.to_datetime(cbr["datetime"], utc=True).dt.tz_convert(TZ)
rates["datetime"] = pd.to_datetime(rates["datetime"], format="mixed", utc=True).dt.tz_convert(TZ)
interfax["datetime"] = pd.to_datetime(interfax["datetime"]).dt.tz_localize(TZ)

cbr["record_id"] = "cbr_doc_" + cbr["doc_id"].astype(str)
rates["record_id"] = "cbr_rate_" + rates["url"].astype(str)
interfax["record_id"] = "interfax_" + interfax["url"].astype(str)

cbr["topics"] = cbr["title"].apply(classify_news).replace("", "cbr_other")
rates["topics"] = "macro"

cbr["event_type"] = "cbr_publication"
rates["event_type"] = "key_rate_decision"
interfax["event_type"] = "news"

cbr["time_source"] = "cbr_doc_date"
rates["time_source"] = "cbr_filename"
interfax["time_source"] = "interfax_archive"

FINAL_COLUMNS = [
    "record_id", "datetime", "title", "url", "source",
    "topics", "event_type", "time_source", "doc_id",
]
for frame in (cbr, rates, interfax):
    for name in FINAL_COLUMNS:
        if name not in frame.columns:
            frame[name] = pd.NA

news_final = (
    pd.concat([cbr[FINAL_COLUMNS], rates[FINAL_COLUMNS], interfax[FINAL_COLUMNS]],
              ignore_index=True)
    .drop_duplicates("record_id")
    .sort_values("datetime")
    .reset_index(drop=True)
)
news_final.to_csv(DATA_DIR / "news_final.csv", index=False, encoding="utf-8-sig")

### 5.1. Примитивная проверка данных

Проверяем число записей, уникальность ID и диапазон дат. 

In [16]:
print("Котировки:", len(stocks), "строк,", stocks["ticker"].nunique(), "тикеров")
print("Все новости Интерфакса:", len(news))
print("Отбор Интерфакса:", len(interfax_filtered))
print("ЦБ:", len(cbr_news), "публикаций +", len(df_rates), "решений")
print("Итоговых новостей:", len(news_final))
print("Дубликаты ID:", news_final["record_id"].duplicated().sum())
print("Пустые даты:", news_final["datetime"].isna().sum())
print("Диапазон:", news_final["datetime"].min(), "—", news_final["datetime"].max())
print("По источникам:")
print(news_final["source"].value_counts())

Котировки: 101008 строк, 19 тикеров
Все новости Интерфакса: 14220
Отбор Интерфакса: 3727
ЦБ: 1121 публикаций + 9 решений
Итоговых новостей: 4857
Дубликаты ID: 0
Пустые даты: 0
Диапазон: 2025-10-24 00:03:00+03:00 — 2026-10-23 19:19:00+03:00
По источникам:
source
Interfax    3727
CBR         1130
Name: count, dtype: int64


**Финальные файлы с котировками и новостями: ** `data/stocks.csv` и `data/news_final.csv`. Исходные файлы с разбивками по месяцам тоже лежат рядом в папках.